## SemEval 2022 Task 2A: Multilingual Idiomaticity Detection - One-Shot Baseline

## Task Overview
- **Objective**: Binary classification of Multi-Word Expressions (MWEs) as idiomatic (0) or literal (1)
- **Languages**: English (EN) and Portuguese (PT)
- **Model**: bert-base-multilingual-cased (mBERT)
- **Setting**: One-shot (1 positive + 1 negative example of each MWE in training)
- All hyperparameters match official implementation

## Key Differences from Zero-Shot:
1. **Training Data**: Uses BOTH train_zero_shot.csv + train_one_shot.csv
2. **Context**: No context (only Target sentence, exclude Previous/Next) for baseline
3. **Input Format**: Two sentences (sentence1=Target, sentence2=MWE) for baseline
4. **Hyperparameters**: Same as zero-shot (matches official baseline)

### 1. Setup and Installation


In [ ]:
# Install dependencies
!pip install -q torch transformers pandas scikit-learn sentencepiece datasets


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
from datasets import Dataset

# Set seed for reproducibility 
SEED = 0
set_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# unzip data if not mounting drive
# !unzip /content/SubTaskA.zip

### 2. Data Loading and Pre-processing

### One-Shot Data Strategy:
- **Load BOTH** train_zero_shot.csv AND train_one_shot.csv
- **Combine** them into single training set
- **Format**: Two-sentence input
  - sentence1 = Target sentence (containing the MWE)
  - sentence2 = The MWE itself (explicit highlighting)

In [ ]:
# Configurations for ablation study
CONTEXT_MODE = "NO_CONTEXT"  # Options: "NO_CONTEXT" (baseline), "WITH_CONTEXT"
USE_SENTENCE2 = True         # Options: True (baseline with MWE), False (single sentence)

# Colab paths
INPUT_DIR = Path("/content/SubTaskA/SubTaskA/Data")
context_label = CONTEXT_MODE
sentence2_label = "WithMWE" if USE_SENTENCE2 else "NoMWE"
OUTPUT_DIR = Path(f"models/OneShot/{context_label}_{sentence2_label}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Context Mode: {CONTEXT_MODE}")
print(f"Use Sentence2 (MWE): {USE_SENTENCE2}")
print(f"Output Directory: {OUTPUT_DIR}")

In [ ]:
# Defining functions to build input sentences based on configuration
# Each function takes a row of the DataFrame and returns sentence1 and sentence2 (or None for single-sentence format)
def build_no_context_with_mwe(row):
    """BASELINE: Target only + MWE as sentence2 (Official one-shot)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    mwe = str(row['MWE']).strip() if pd.notna(row['MWE']) else ""
    return target, mwe

def build_with_context_with_mwe(row):
    """ABLATION 1: Full context + MWE as sentence2"""
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    context = f"{prev} {target} {nxt}".strip()
    mwe = str(row['MWE']).strip() if pd.notna(row['MWE']) else ""
    return context, mwe

def build_no_context_no_mwe(row):
    """ABLATION 2: Target only, single sentence (no MWE separation)"""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    return target, None  # None indicates single-sentence format

def build_with_context_no_mwe(row):
    """ABLATION 3: Full context, single sentence (zero-shot format on one-shot data)"""
    prev = str(row['Previous']).strip() if pd.notna(row['Previous']) else ""
    target = str(row['Target']).strip() if pd.notna(row['Target']) else ""
    nxt = str(row['Next']).strip() if pd.notna(row['Next']) else ""
    context = f"{prev} {target} {nxt}".strip()
    return context, None 

# Map builder based on configuration
if CONTEXT_MODE == "NO_CONTEXT" and USE_SENTENCE2:
    build_function = build_no_context_with_mwe
    description = "BASELINE: Target only + MWE as sentence2"
elif CONTEXT_MODE == "WITH_CONTEXT" and USE_SENTENCE2:
    build_function = build_with_context_with_mwe
    description = "ABLATION: Full context + MWE as sentence2"
elif CONTEXT_MODE == "NO_CONTEXT" and not USE_SENTENCE2:
    build_function = build_no_context_no_mwe
    description = "ABLATION: Target only, single sentence"
else:  # WITH_CONTEXT and not USE_SENTENCE2
    build_function = build_with_context_no_mwe
    description = "ABLATION: Full context, single sentence (zero-shot format)"

In [ ]:
# Load and Preprocess Data
print("\nLoading datasets")

# Load zero-shot training data
train_zero_df = pd.read_csv(INPUT_DIR / "train_zero_shot.csv")
print(f"Zero-shot train: {len(train_zero_df)} samples")

# Load one-shot training data
train_one_df = pd.read_csv(INPUT_DIR / "train_one_shot.csv")
print(f"One-shot train: {len(train_one_df)} samples")

# Combine both datasets
train_combined = pd.concat([train_zero_df, train_one_df], ignore_index=True)
print(f"Combined train: {len(train_combined)} samples")

# Build sentence pairs using the selected builder function
sentence_pairs = train_combined.apply(build_function, axis=1)

train_data = pd.DataFrame({
    "label": train_combined["Label"],
    "sentence1": [pair[0] for pair in sentence_pairs],  # Target sentence or context
    "sentence2": [pair[1] if pair[1] is not None else "" for pair in sentence_pairs]  # MWE or empty
})

# Load dev data
dev_df = pd.read_csv(INPUT_DIR / "dev.csv")
dev_gold = pd.read_csv(INPUT_DIR / "dev_gold.csv")
dev_df = dev_df.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Build sentence pairs for dev using the same builder function
dev_sentence_pairs = dev_df.apply(build_function, axis=1)

dev_data = pd.DataFrame({
    "label": dev_df["Label"],
    "sentence1": [pair[0] for pair in dev_sentence_pairs],
    "sentence2": [pair[1] if pair[1] is not None else "" for pair in dev_sentence_pairs]
})

print(f"Dev: {len(dev_data)} samples")

# Distribution analysis
print("\nDataset Statistics")
print(f"\nTraining Set:")
print(f"  Total: {len(train_data)} samples")
print(f"  Label 0 (Idiomatic): {(train_data['label']==0).sum()} ({(train_data['label']==0).sum()/len(train_data)*100:.1f}%)")
print(f"  Label 1 (Literal): {(train_data['label']==1).sum()} ({(train_data['label']==1).sum()/len(train_data)*100:.1f}%)")

print(f"\nDev Set:")
print(f"  Total: {len(dev_data)} samples")
print(f"  Label 0 (Idiomatic): {(dev_data['label']==0).sum()} ({(dev_data['label']==0).sum()/len(dev_data)*100:.1f}%)")
print(f"  Label 1 (Literal): {(dev_data['label']==1).sum()} ({(dev_data['label']==1).sum()/len(dev_data)*100:.1f}%)")

### 3. Model Parameters

#### Fixed hyperparameters as used in Baseline
- Model: `bert-base-multilingual-cased`
- Learning rate: `2e-5`
- Batch size: `32`
- Epochs: `9`
- Max sequence length: `128`
- Seed: `0`

In [ ]:
MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128

print(f"Loading model and tokenizer: {MODEL_NAME}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model (num_labels=2 for binary classification)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Note: Warning about "classifier weights not initialized" is EXPECTED
# The warning is because we are loading a pre-trained model without a classification head 
# The classification head is randomly initialized and will be trained

In [ ]:
# Tokenisation: it takes raw text and converts it into input IDs and attention masks.
'''BERT will receive: [CLS] sentence1 [SEP] sentence2 [SEP] '''

def tokenize_function_pair(examples):
    """
    Adaptive tokenization for ablations:
    - If sentence2 is provided (not empty): Two-sentence format [CLS] sentence1 [SEP] sentence2 [SEP]
    - If sentence2 is empty: Single-sentence format [CLS] sentence1 [SEP]
    """
    # Check if we have sentence2 values
    has_sentence2 = all(s.strip() != "" for s in examples["sentence2"])
    
    if has_sentence2:
        # Two-sentence format (baseline and context+MWE ablations)
        return tokenizer(
            examples["sentence1"],
            examples["sentence2"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )
    else:
        # Single-sentence format (no-MWE ablations)
        return tokenizer(
            examples["sentence1"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH
        )

print("Tokenizing datasets")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_data)
dev_dataset = Dataset.from_pandas(dev_data)

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function_pair, batched=True)
dev_dataset = dev_dataset.map(tokenize_function_pair, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])
dev_dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

print(f" Train dataset tokenized: {len(train_dataset)} samples")
print(f" Dev dataset tokenized: {len(dev_dataset)} samples")


### 4. TRAINING

Training with fixed hyperparameters matching the official baseline.
- Early stopping enabled based on dev set F1 score
- Best checkpoint saved automatically

In [ ]:
# Metric computation function
def compute_metrics(eval_pred):
    """Calculate Macro F1 score (average of F1 for each class)"""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"f1": macro_f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    
    # Hyperparameters (Baseline replication hence fixed)
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=9,
    
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro-F1",
    
    # Reproducibility
    seed=SEED,
    
    # Logging
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Training configuration:\n")
print(f"Model: {MODEL_NAME}")
print(f"Context mode: {CONTEXT_MODE}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Max length: {MAX_LENGTH}")
print(f"Seed: {SEED}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
print("Model Training in Progress\n")

# Train model
train_result = trainer.train()

print("\n Training complete!")
print(f"Best model checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best Macro-F1 score: {trainer.state.best_metric:.4f}")

### 5. Evaluation
Analysing results with respect to languages and classes in dataset
- Class (Idiomatic vs Literal)
- Language (EN vs PT)
- Combined metrics

In [ ]:
print("Generating predictions on dev set...")

# Get predictions
predictions = trainer.predict(dev_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)

# Load language information for detailed analysis
dev_full = pd.read_csv(INPUT_DIR / "dev.csv")
dev_full = dev_full.merge(dev_gold[["ID", "Label"]], on="ID", how="left")

# Create results dataframe
results_df = pd.DataFrame({
    'ID': dev_full['ID'],
    'Language': dev_full['Language'],
    'Gold_Label': dev_full['Label'],
    'Predicted_Label': pred_labels
})

print(f"Predictions generated: {len(results_df)} samples")

In [ ]:
print(f"Evaluation results")

print(f"\nTotal samples: {len(results_df)}")
print(f"Languages: {sorted(results_df['Language'].unique())}")
print(f"Classes: 0 (Idiomatic), 1 (Literal)")
print("-"*80)

# Results for combinened classes
print("\nOverall Results")
print("\n" + classification_report(
    results_df['Gold_Label'], 
    results_df['Predicted_Label'],
    target_names=['Idiomatic', 'Literal'],
    digits=4
))

# Macro-F1 for combined classes
macro_f1 = f1_score(results_df['Gold_Label'], results_df['Predicted_Label'], average='macro')

print("-"*80)
print("\nResults by Language")
for lang in sorted(results_df['Language'].unique()):
    lang_data = results_df[results_df['Language'] == lang]
    
    print(f"\n{lang} ({len(lang_data)} samples)")
    print(classification_report(
        lang_data['Gold_Label'], 
        lang_data['Predicted_Label'],
        target_names=['Idiomatic', 'Literal'],
        digits=4
    ))

# Calculate macro F1 for each language
en_data = results_df[results_df['Language'] == 'EN']
pt_data = results_df[results_df['Language'] == 'PT']

en_macro_f1 = f1_score(en_data['Gold_Label'], en_data['Predicted_Label'], average='macro')
pt_macro_f1 = f1_score(pt_data['Gold_Label'], pt_data['Predicted_Label'], average='macro')

print("-"*80)
print(f"\nSummary of Macro F1 Scores")
summary_data = [
    {'Metric': 'EN Macro F1', 'Value': f"{en_macro_f1:.4f}"},
    {'Metric': 'PT Macro F1', 'Value': f"{pt_macro_f1:.4f}"},
    {'Metric': 'Combined Macro F1', 'Value': f"{macro_f1:.4f}"}
]
summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

---
## 7. ABLATION STUDIES - SYSTEMATIC EXPERIMENTATION

### Instructions for Running Ablations:

To test different configurations:

1. **Go back to: "CONFIGURATION - ABLATION STUDY SETTINGS"**
2. **Change settings:**
   - `CONTEXT_MODE`: "NO_CONTEXT" or "WITH_CONTEXT"
   - `USE_SENTENCE2`: True or False

3. **Re-run all cells from Section 2 onwards**

4. **Record results in the table below**

### Expected Ablation Results Template:

| Context | Sentence2 | EN F1 | PT F1 | Combined F1 | Notes |
|---------|-----------|-------|-------|-------------|-------|
| NO_CONTEXT | True (MWE) | ?.???? | ?.???? | ?.???? | **BASELINE** - Official one-shot |
| WITH_CONTEXT | True (MWE) | ?.???? | ?.???? | ?.???? | Does context help with MWE? |
| NO_CONTEXT | False | ?.???? | ?.???? | ?.???? | Is MWE highlighting necessary? |
| WITH_CONTEXT | False | ?.???? | ?.???? | ?.???? | Zero-shot format on one-shot data |

### Research Questions:
- **Q1:** Does adding context improve one-shot performance?
- **Q2:** How much does explicitly highlighting the MWE (sentence2) help?
- **Q3:** Can zero-shot input format benefit from additional one-shot training data?
- **Q4:** Is there an interaction between context and MWE highlighting?

### Expected Patterns:
- Baseline (NO_CONTEXT + MWE): ~0.82 F1
- +Context might improve: better disambiguation
- -Sentence2 will likely drop: loses explicit MWE focus
- WITH_CONTEXT + No MWE: interesting hybrid approach

---
## 📝 NOTES

### One-Shot vs Zero-Shot Comparison:

| Aspect | Zero-Shot | One-Shot |
|--------|-----------|----------|
| Training Data | train_zero_shot.csv only | BOTH files |
| Context | ✅ Previous + Target + Next | ❌ Target only |
| Sentence2 | ❌ No | ✅ MWE |
| Input Format | Single sentence | Two sentences |
| token_type_ids | All 0s | 0s and 1s |
| Difficulty | 🔴 Harder | 🟡 Easier |

### Expected Performance:
- One-shot typically performs BETTER than zero-shot (model has seen examples of each MWE)
- Expected F1: ~0.80-0.85 (higher than zero-shot's ~0.75-0.80)

### Reproducibility Checklist:
- ✅ Seed fixed at 0 (matches official baseline)
- ✅ All hyperparameters match official implementation
- ✅ Same model: bert-base-multilingual-cased
- ✅ Same data: Both zero-shot + one-shot files
- ✅ Same input format: Two sentences
- ✅ Same evaluation metric: Macro F1

### For Research Paper:
```
We replicated the official one-shot baseline using bert-base-multilingual-cased
with fixed hyperparameters (lr=2e-5, batch=32, epochs=9, seed=0).
The model was trained on both zero-shot and one-shot data, using a two-sentence
input format (Target sentence + MWE) without surrounding context, and achieved
[X.XX] Macro F1 on the development set.
```